# Preprocessing — DROZY (Clip Generation + KSS Labelling)

Generates standardised video clips for the DROZY external test set and assigns drowsiness labels from the Karolinska Sleepiness Scale (KSS). All clips are extracted with ffmpeg at **10 fps, 112×112 resolution, 16-frame length, stride 8**, then saved as `.npz` arrays. KSS scores ≥ 6 are labelled drowsy, following the dissertation's labelling protocol (Section 3.5).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install numpy pandas tqdm
!apt-get -qq update
!apt-get -qq install -y ffmpeg

In [ ]:
script = r'''
import argparse
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

def list_videos(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def ffmpeg_extract(video_path, fps, size):
    cmd = [
        "ffmpeg", "-v", "error",
        "-i", str(video_path),
        "-vf", f"fps={fps},scale={size}:{size}",
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "-"
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE)
    raw = p.stdout
    frame_bytes = size * size * 3
    if len(raw) < frame_bytes:
        return np.zeros((0, size, size, 3), dtype=np.uint8)

    nframes = len(raw) // frame_bytes
    raw = raw[:nframes * frame_bytes]
    frames = np.frombuffer(raw, dtype=np.uint8).reshape(nframes, size, size, 3)
    return frames

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_dir", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--target_fps", type=float, default=10)
    ap.add_argument("--clip_len", type=int, default=16)
    ap.add_argument("--stride", type=int, default=8)
    ap.add_argument("--size", type=int, default=112)
    args = ap.parse_args()

    in_root = Path(args.input_dir)
    out_root = Path(args.output_dir)
    out_clips = out_root / "clips_npz"
    safe_mkdir(out_clips)

    videos = list_videos(in_root)
    rows = []

    for vp in tqdm(videos, desc="Processing"):
        try:
            frames = ffmpeg_extract(vp, args.target_fps, args.size)
            T = frames.shape[0]

            if T < args.clip_len:
                rows.append({"video": str(vp), "error": "too_few_frames"})
                continue

            clip_idx = 0
            for start in range(0, T - args.clip_len + 1, args.stride):
                clip = frames[start:start+args.clip_len]
                clip = np.transpose(clip, (3,0,1,2))
                out_path = out_clips / f"{vp.stem}__{clip_idx:05d}.npz"
                np.savez_compressed(out_path, clip=clip)
                clip_idx += 1

            rows.append({"video": str(vp), "clips_written": clip_idx})

        except Exception as e:
            rows.append({"video": str(vp), "error": str(e)})

    out_root.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_root / "manifest.csv", index=False)
    print("Done.")

if __name__ == "__main__":
    main()
'''
open("preprocess_ffmpeg.py","w").write(script)
print("Created preprocess_ffmpeg.py")

In [ ]:
INPUT_DIR  = "/content/drive/MyDrive/ES327_Drowsiness/Datasets_Preprocessed/DROZY_mp4_clean"
OUTPUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8"
DATASET    = "DROZY"

!python preprocess_ffmpeg.py \
  --input_dir "$INPUT_DIR" \
  --output_dir "$OUTPUT_DIR" \
  --dataset "$DATASET" \
  --target_fps 10 \
  --clip_len 16 \
  --stride 8 \
  --size 112

In [ ]:
import os, glob

OUT = OUTPUT_DIR  # from your notebook
clips_dir = os.path.join(OUT, "clips_npz")
clips = sorted(glob.glob(clips_dir + "/**/*.npz", recursive=True))

print("clips_dir:", clips_dir)
print("num .npz clips:", len(clips))
print("example files:", clips[:5])

In [ ]:
import numpy as np

sample = clips[0]
x = np.load(sample)["clip"]
print("sample:", sample)
print("shape:", x.shape, "dtype:", x.dtype, "min/max:", x.min(), x.max())

In [ ]:
import pandas as pd, os
manifest = os.path.join(OUT, "manifest.csv")
df = pd.read_csv(manifest)
print("manifest rows:", len(df))
print(df.head())

# show any errors if present
if "error" in df.columns:
    errs = df[df["error"].notna()]
    print("errors:", len(errs))
    display(errs.head(20))

In [ ]:
import os, glob, shutil, re
from tqdm import tqdm

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8"
src = os.path.join(OUT, "clips_npz")
dst = os.path.join(OUT, "clips_by_subject")
os.makedirs(dst, exist_ok=True)

clips = sorted(glob.glob(src + "/*.npz"))
print("Found clips:", len(clips))

bad = []
for fp in tqdm(clips, desc="Copying by subject"):
    base = os.path.basename(fp)          # e.g. 1-1__00000.npz
    video_stem = base.split("__")[0]     # e.g. 1-1
    m = re.match(r"^(\d+)-", video_stem)
    if not m:
        bad.append(base)
        continue

    sid = int(m.group(1))
    subdir = os.path.join(dst, f"sub_{sid:02d}")
    os.makedirs(subdir, exist_ok=True)

    shutil.copy2(fp, os.path.join(subdir, base))

print("Done.")
print("Bad filenames:", len(bad))
if bad:
    print("Examples:", bad[:10])

In [ ]:
import os, glob

dst = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8/clips_by_subject"
subs = sorted([d for d in os.listdir(dst) if d.startswith("sub_")])
print("Subject folders:", len(subs))
print("Folders:", subs)

for s in subs:
    n = len(glob.glob(os.path.join(dst, s, "*.npz")))
    print(s, n)

In [ ]:
import os

root = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8/clips_by_subject"
print(sorted(os.listdir(root)))

In [ ]:
# =========================
# 1) Build DROZY manifest with subject + recording + clip index
# =========================
import os, glob, re
import pandas as pd

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8"
clips_dir = os.path.join(OUT, "clips_npz")  # flat folder of .npz clips (fast)
clips = sorted(glob.glob(os.path.join(clips_dir, "*.npz")))

rows = []
# matches either:
#   1-1__00000.npz
#   1-1__clip00000.npz
pat = re.compile(
    r"^(?P<rec>\d+-\d+)__clip(?P<clip>\d+)\.npz$|^(?P<rec2>\d+-\d+)__(?P<clip2>\d+)\.npz$"
)

bad = 0
for fp in clips:
    base = os.path.basename(fp)
    m = pat.match(base)
    if not m:
        bad += 1
        continue

    rec = m.group("rec") or m.group("rec2")           # e.g. "1-1"
    clip_idx = m.group("clip") or m.group("clip2")    # e.g. "00000"
    subj = rec.split("-")[0]                          # e.g. "1"

    rows.append({
        "dataset": "DROZY",
        "subject_id": int(subj),
        "recording_id": rec,          # e.g. "1-1"
        "clip_idx": int(clip_idx),
        "clip_path": fp
    })

df = pd.DataFrame(rows).sort_values(["subject_id","recording_id","clip_idx"]).reset_index(drop=True)

out_csv = os.path.join(OUT, "manifest_drozy_subject_recording.csv")
df.to_csv(out_csv, index=False)

print("✅ Wrote:", out_csv)
print("Rows:", len(df), "Bad filenames:", bad)
display(df.head())

In [ ]:
# =========================
# 2) Summary tables (non-uniformity check)
# =========================
print("Subjects:", df["subject_id"].nunique())
print("Recordings:", df["recording_id"].nunique())

summary = (
    df.groupby(["subject_id","recording_id"])
      .size()
      .reset_index(name="num_clips")
      .sort_values(["subject_id","recording_id"])
)
display(summary.head(30))

subj_totals = df.groupby("subject_id").size().reset_index(name="total_clips")
display(subj_totals.sort_values("total_clips"))

In [ ]:
import glob

flat = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8/clips_npz"
print("Flat clips count:", len(glob.glob(flat + "/*.npz")))

In [ ]:
import pandas as pd
import re

OUT = "/content/drive/MyDrive/ES327_Drowsiness/Clips/DROZY_112_10fps_16f_stride8"
manifest_in = f"{OUT}/manifest_drozy_subject_recording.csv"
df = pd.read_csv(manifest_in)

df.head()

In [ ]:
# KSS triplets per subject 1..14 in order: [session1, session2, session3]
kss_rows = [
    [3, 6, 7],
    [3, 7, 6],
    [2, 3, 4],
    [4, 8, 9],
    [3, 7, 8],
    [2, 3, 7],
    [0, 4, 9],
    [2, 6, 8],
    [2, 6, 8],
    [3, 6, 7],
    [4, 7, 7],
    [2, 5, 6],
    [6, 3, 7],
    [5, 7, 8],
]

assert len(kss_rows) == 14
for r in kss_rows:
    assert len(r) == 3

# Build dict: (subject_id, session_idx) -> kss
kss_map = {}
for sid in range(1, 15):
    for sess in (1, 2, 3):
        kss_map[(sid, sess)] = int(kss_rows[sid-1][sess-1])

In [ ]:
def parse_session_idx(rec_id: str) -> int:
    # expects "subject-session", e.g. "12-3"
    m = re.match(r"^\d+-(\d+)$", str(rec_id))
    if not m:
        return -1
    return int(m.group(1))

df["session_idx"] = df["recording_id"].apply(parse_session_idx)

In [ ]:
def lookup_kss(row):
    key = (int(row["subject_id"]), int(row["session_idx"]))
    return kss_map.get(key, None)

df["kss"] = df.apply(lookup_kss, axis=1)

# Main label (recommended): drowsy if KSS >= 6
df["label_kss_ge6"] = (df["kss"] >= 6).astype(int)

# Stricter label (optional): drowsy if KSS >= 7
df["label_kss_ge7"] = (df["kss"] >= 7).astype(int)

print("Missing KSS rows:", df["kss"].isna().sum())
df[["subject_id","recording_id","session_idx","kss","label_kss_ge6","label_kss_ge7"]].head(10)

In [ ]:
out_csv = f"{OUT}/manifest_drozy_kss_labeled.csv"
df.to_csv(out_csv, index=False)
print("✅ Wrote:", out_csv, "rows:", len(df))

In [ ]:
print("Label distribution (KSS>=6):")
print(df["label_kss_ge6"].value_counts())

print("\nLabel distribution (KSS>=7):")
print(df["label_kss_ge7"].value_counts())

print("\nPer-subject positives (KSS>=6):")
print(df.groupby("subject_id")["label_kss_ge6"].mean().sort_values())

In [ ]:
# recordings that exist in clips
existing_recs = set(df["recording_id"].unique())

# recordings defined in KSS map (excluding None entries)
defined_kss_recs = set()
for (sid, sess), val in kss_map.items():
    if val is not None:
        defined_kss_recs.add(f"{sid}-{sess}")

print("Recordings in files but NOT in KSS map:")
print(sorted(existing_recs - defined_kss_recs))

print("\nRecordings in KSS map but NOT in files:")
print(sorted(defined_kss_recs - existing_recs))